In [ ]:
import tensorflow as tf

from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

# Verify GPU is available
print(tf.config.list_physical_devices('GPU'))

# Mixed precision: uses float16 on GPU for ~2x speedup on Ada/Ampere GPUs
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# XLA JIT compilation for additional ~10-20% speedup
tf.config.optimizer.set_jit(True)

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    # The CIFAR labels happen to be arrays, 
    # which is why you need the extra index
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

In [ ]:
inputs = layers.Input(shape=(32, 32, 3))

# Augmentation
x = layers.RandomFlip("horizontal")(inputs)              # FLIP AUGMENTATION
x = layers.RandomTranslation(0.125, 0.125, fill_mode='reflect')(x)  # SHIFT AUGMENTATION

# Block 1
x = layers.Conv2D(8, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Conv2D(16, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Block 2 — with skip connection (projection shortcut 16 → 32)
skip = x
skip = layers.Conv2D(32, (1, 1), padding='same')(skip)
skip = layers.BatchNormalization()(skip)

x = layers.Conv2D(32, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

x = layers.Conv2D(32, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Add()([x, skip])                             # SKIP CONNECTION
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Block 3 — with skip connection (projection shortcut 32 → 64)
skip = x
skip = layers.Conv2D(64, (1, 1), padding='same')(skip)
skip = layers.BatchNormalization()(skip)

x = layers.Conv2D(64, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

x = layers.Conv2D(64, (3, 3), padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Add()([x, skip])                             # SKIP CONNECTION
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Classifier
x = layers.GlobalAveragePooling2D()(x)                  # GLOBAL AVERAGE POOLING
x = layers.Dropout(0.1)(x)
# dtype='float32' keeps output numerically stable with mixed precision
outputs = layers.Dense(10, dtype='float32')(x)

model = models.Model(inputs, outputs)
model.summary()

In [ ]:
# Model is fully defined in the cell above (Functional API with skip connection)

In [ ]:
BATCH_SIZE = 256

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_images, train_labels))
    .shuffle(50000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

history = model.fit(train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop])

In [ ]:
plt.plot(history.history['sparse_categorical_crossentropy'], label='sparse_categorical_crossentropy')
plt.plot(history.history['val_sparse_categorical_crossentropy'], label = 'val_sparse_categorical_crossentropy')
plt.xlabel('Epoch')
plt.ylabel('CE')
plt.ylim([0.6, 3.5])
plt.legend(loc='lower right')

test_loss, test_ce = model.evaluate(test_images,  test_labels, verbose=2)

In [ ]:
print(test_ce)